# NEURAL LANGUAGE MODELS

Adapted implementation of Neural Language Model proposed in paper *"A Neural Probabilistic Language Model"* (Bengio et al., 2003).  
[https://www.jmlr.org/papers/volume3/bengio03a/bengio03a.pdf](https://www.jmlr.org/papers/volume3/bengio03a/bengio03a.pdf)

In [45]:
#Tools 
import os 
import time
import random
from typing import Tuple
from argparse import Namespace
import matplotlib.pyplot as plt
import shutil

#Preprocessing
import nltk
from nltk.corpus import stopwords 
from nltk import ngrams
from nltk.tokenize import TweetTokenizer
from nltk import FreqDist
import pandas as pd
import numpy as np

#PyTorch
from torch.utils.data import DataLoader, TensorDataset
import torch 
import torch.nn as nn
import torch.nn.functional as F

#scikit-learn
from sklearn.metrics import accuracy_score

In [16]:
seed = 42
random.seed(seed) #python seed
np.random.seed(seed) #numpy seed
torch.manual_seed(seed) #  torch seed
torch.backends.cudnn.benchmark = False #sets a deterministic and stable implementation of settings

In [17]:
X_train = pd.read_csv('mex20_train.txt',sep='\r\n',engine='python',header=None).loc[:,0].values.tolist()
print(X_train)

X_val = pd.read_csv('mex20_val.txt',sep='\r\n',engine='python',header=None).loc[:,0].values.tolist()
print(X_val)

['@USUARIO @USUARIO @USUARIO Q se puede esperar del maricon de closet de la Yañez aun recuerdo esa ves q lo vi en zona rosa viendo quien lo levantada', '@USUARIO La piel nueva siempre arde un poquito los primeros días... y más con este puto clima', 'Ustedes no se enamoran de mí… por tontas.', 'Me las va a pagar esa puta gorda roba tuits...', '@USUARIO LA GENTE ES TONTA PORQUE NO SE DAN CUENTA QUE TÚ HACES A BATMAN AZUL', 'Estoy muy encabronada con las pseudo feministas por tontas e iletradas, a veces me avergüenza ser mujer; preferiría tener un falo. #NiUnaMas', 'Anden putos, recuerdan el #noerapenal #Holanda fuera de #Rusia2018, esto se llama #karma ehhhhhhhh #puuuuuutos', 'Si no tienen chichis no traten de enseñar se ven muy mal y más cuando son prietas.', 'Ojalá asi me agarrars cuando te digo que me voy en lugar de correrme a la verga cada 5 minutos.', '@USUARIO @USUARIO @USUARIO @USUARIO Es solo un HDP aprovechado y que su "Diosito Bimbo" me perdone', 'La próxima vez que diga que m

In [18]:
args=Namespace()
args.N = 4


In [19]:
class NgramData():
    
    def __init__(self, N: int, vocab_max: int=5000, tokenizer=None, embeddings_model=None):
        self.tokenizer = tokenizer if tokenizer else self.default_tokenizer
        self.punct = set(['.', ',', ';', ':', '-','^','»','!','¡','¿','?','"','\'','...','<url>','*','@usuario'])
        
        self.N = N # n-gram size
        self.vocab_max = vocab_max
        self.UNK ="<unk>"
        self.SOS = '<s>' # Start of sentence
        self.EOS = '</s>'
        self.embeddings_model = embeddings_model
        
        
    def get_vocab_size(self) -> int:
        return len(self.vocab)

        
    def default_tokenizer(self, doc: str) -> list:
        return doc.split(" ")
    
    def remove_word(self, word: str) -> bool:
        word = word.lower()
        is_punct = True if word in self.punct else False
        is_digit = word.isnumeric()
        
        return is_punct or is_digit
    
    def get_vocab(self, corpus: list) -> set:
        freq_dist = FreqDist([w.lower() for sentence in corpus\
            for w in self.tokenizer(sentence)\
            if not self.remove_word(w)])
            
            
        sorted_words = self.sortFreqDict(freq_dist)[:self.vocab_max-3]
        return set(sorted_words)
    
    def sortFreqDict(self, freq_dist) ->list:
        freq_dict = dict(freq_dist)
        return sorted(freq_dict, key=freq_dict.get, reverse=True)
    
    def fit(self, corpus: list) -> None:
        self.vocab = self.get_vocab(corpus)
        self.vocab.add(self.UNK)
        self.vocab.add(self.SOS)
        self.vocab.add(self.EOS) 
        
        
        self.w2id = {}
        self.id2w = {}
        
        if self.embeddings_model is not None:
            self.embeddings_matrix = np.empty([len(self.vocab), self.embeddings_model.vector_size])
            
            
        id = 0
        for doc in corpus: 
            for word in self.tokenizer(doc):
                word_ = word.lower()
                if word_ in self.vocab and not word_ in self.w2id:
                    self.w2id[word_] = id
                    self.id2w[id] = word_
                    
                    
                    if self.embeddings_model is not None:
                        if word_ in self.embeddings_model:
                            self.embeddings_matrix[id] = self.embeddings_model[word_]
                        else:
                            self.embeddings_matrix[id] = np.random.rand(self.embeddings_model.vector_size)
                
                
                    id += 1
                
        
        # Always add special tokens
        self.w2id.update(
            {
                self.UNK: id,
                self.SOS: id + 1,
                self.EOS: id + 2
            }
        )

        self.id2w.update(
            {
                id: self.UNK,
                id + 1: self.SOS,
                id + 2: self.EOS
            }
        )
        
    
    def transform(self, corpus: list) -> Tuple[np.ndarray, np.ndarray]:
        X_ngrams = []
        y = []

        for doc in corpus:
            doc_ngram = self.get_ngram_doc(doc)
            for words_window in doc_ngram:
                words_window_ids = [self.w2id[w] for w in words_window]
                X_ngrams.append(list(words_window_ids[:-1]))
                y.append(words_window_ids[-1])

        return np.array(X_ngrams), np.array(y)
    
    
    def get_ngram_doc(self, doc: str) -> list:
        doc_tokens = self.tokenizer(doc)
        doc_tokens = self.replace_unk(doc_tokens)
        doc_tokens = [w.lower() for w in doc_tokens]
        doc_tokens = [self.SOS] * (self.N - 1) + doc_tokens + [self.EOS]
        
        return list(ngrams(doc_tokens, self.N))
    
    
    def replace_unk(self, doc_tokens: list) -> list:
        for i, token in enumerate(doc_tokens):
            if token.lower() not in self.vocab:
                doc_tokens[i] = self.UNK
                    
        return doc_tokens


In [21]:
tk = TweetTokenizer()
ngram_data = NgramData(args.N,5000,tk.tokenize)
ngram_data.fit(X_train)

In [ ]:
print(f'Vocab size: {ngram_data.get_vocab_size()}')

In [22]:
ngram_data.vocab

{'tía',
 'laptop',
 'carajo',
 '#chivas',
 'bato',
 'entendido',
 'placer',
 '#puebla',
 'wey',
 'mande',
 'mota',
 '👋🏼',
 'tema',
 '/',
 'despierten',
 'huevo',
 'único',
 'favorita',
 'chinguen',
 'piernas',
 'maña',
 'quiso',
 'dijeran',
 'perdida',
 'ignorancia',
 'intensa',
 'premios',
 'ponganse',
 'señal',
 'semana',
 'aczino',
 'sobre',
 'fea',
 'materiales',
 'ferrari',
 'migajas',
 '❤',
 'veías',
 'valiendo',
 'aman',
 'yeah',
 'casas',
 'voces',
 'líder',
 'mentadas',
 '🤦🏼\u200d♀',
 'modelos',
 'trague',
 'márquez',
 'gif',
 'grité',
 'llego',
 'agresivo',
 'perros',
 'palo',
 'quedó',
 'princesa',
 'toma',
 'ridiculo',
 'pregunto',
 '⚡',
 'escribo',
 'twitter',
 'encantan',
 'harto',
 '🎄',
 'vimos',
 'drogas',
 'fué',
 'chavo',
 'tal',
 'pendejas',
 'tono',
 'prensa',
 'anda',
 'reyna',
 'miserable',
 'bai',
 'esteroides',
 'verte',
 'total',
 'adivinen',
 'sur',
 'tapan',
 'solucionan',
 'playeras',
 '#exo',
 'risas',
 'partes',
 'sirve',
 'sonríe',
 'estan',
 'patas',
 's

In [23]:
X_ngram_train, y_ngram_train = ngram_data.transform(X_train)
X_ngram_val, y_ngram_val = ngram_data.transform(X_val)

In [24]:
X_ngram_train

array([[4998, 4998, 4998],
       [4998, 4998, 4997],
       [4998, 4997, 4997],
       ...,
       [4997,  945,   31],
       [ 945,   31, 2521],
       [  31, 2521, 4997]])

In [25]:
[[ngram_data.id2w[w] for w in tw] for tw in X_ngram_train[:22]]

[['<s>', '<s>', '<s>'],
 ['<s>', '<s>', '<unk>'],
 ['<s>', '<unk>', '<unk>'],
 ['<unk>', '<unk>', '<unk>'],
 ['<unk>', '<unk>', 'q'],
 ['<unk>', 'q', 'se'],
 ['q', 'se', 'puede'],
 ['se', 'puede', 'esperar'],
 ['puede', 'esperar', 'del'],
 ['esperar', 'del', 'maricon'],
 ['del', 'maricon', 'de'],
 ['maricon', 'de', 'closet'],
 ['de', 'closet', 'de'],
 ['closet', 'de', 'la'],
 ['de', 'la', 'yañez'],
 ['la', 'yañez', 'aun'],
 ['yañez', 'aun', 'recuerdo'],
 ['aun', 'recuerdo', 'esa'],
 ['recuerdo', 'esa', 'ves'],
 ['esa', 'ves', 'q'],
 ['ves', 'q', 'lo'],
 ['q', 'lo', 'vi']]

In [26]:
y_ngram_train

array([4997, 4997, 4997, ..., 2521, 4997, 4999])

In [27]:
[ngram_data.id2w[w] for w in y_ngram_train[:22]]

['<unk>',
 '<unk>',
 '<unk>',
 'q',
 'se',
 'puede',
 'esperar',
 'del',
 'maricon',
 'de',
 'closet',
 'de',
 'la',
 'yañez',
 'aun',
 'recuerdo',
 'esa',
 'ves',
 'q',
 'lo',
 'vi',
 'en']

In [28]:
print(f'Training observations: X: {X_ngram_train.shape}, y: {y_ngram_train.shape}')
print(f'Validation observations: X: {X_ngram_val.shape}, y: {y_ngram_val.shape}')

Training observations: X: (102751, 3), y: (102751,)
Validation observations: X: (11558, 3), y: (11558,)


In [ ]:
#Set batch in args 
args.batch_size = 64

# Num workers
args.num_workers = 2

# Train
train_dataset = TensorDataset(torch.tensor(X_ngram_train, dtype=torch.int64),
                              torch.tensor(y_ngram_train, dtype=torch.int64))

train_loader = DataLoader(train_dataset,
                          batch_size=args.batch_size,
                          num_workers = args.num_workers,
                          shuffle=True)

# Val
# Train
val_dataset = TensorDataset(torch.tensor(X_ngram_val, dtype=torch.int64),
                              torch.tensor(y_ngram_val, dtype=torch.int64))

val_loader = DataLoader(val_dataset,
                          batch_size=args.batch_size,
                          num_workers = args.num_workers,
                          shuffle=False)

In [36]:
batch = next(iter(train_loader))
print(f'X shape: {batch[0].shape}')
print(f'y shape: {batch[1].shape}')

X shape: torch.Size([64, 3])
y shape: torch.Size([64])


In [37]:
batch[0]

tensor([[  82, 4997, 4997],
        [4997, 4997,   45],
        [1745,  105, 1027],
        [ 329,  359,   59],
        [4997,  125,  800],
        [ 284,  285,   82],
        [2113, 4997, 4997],
        [  32, 4997, 2255],
        [  93,   26,  254],
        [ 983, 4997,  164],
        [ 138, 4997, 1966],
        [  38,  105,  710],
        [ 462,   92,  710],
        [  82,   47,  341],
        [4997,  131,   38],
        [4998, 4998, 1242],
        [   1, 4997,   42],
        [4997,  622,   47],
        [   0, 1472,   45],
        [4997, 4997,   44],
        [ 192,  351, 4997],
        [4997, 4027, 4997],
        [  44, 1909,    8],
        [  59,  123,   50],
        [ 710,   50, 4997],
        [4998,  251,  139],
        [4998, 4998, 2567],
        [4998, 4998, 4998],
        [  64, 4997,    8],
        [  38,  485,  996],
        [4997,  548,  549],
        [4997, 3541, 4997],
        [ 298,   92,  590],
        [   8,  127, 1700],
        [  80, 3513, 4997],
        [3469,    6,

In [38]:
[[ngram_data.id2w[w] for w in tw] for tw in batch[0].tolist()]

[['el', '<unk>', '<unk>'],
 ['<unk>', '<unk>', 'las'],
 ['uy', 'te', 'vuelves'],
 ['toda', 'persona', 'que'],
 ['<unk>', 'alguien', 'sabe'],
 ['coldplay', 'cerrará', 'el'],
 ['carajo', '<unk>', '<unk>'],
 ['más', '<unk>', 'seria'],
 ['tienen', 'un', 'gusto'],
 ['gracias', '<unk>', 'pero'],
 ['loca', '<unk>', '😨'],
 ['no', 'te', 'hace'],
 ['jajajaja', 'si', 'hace'],
 ['el', 'a', 'mi'],
 ['<unk>', 'ya', 'no'],
 ['<s>', '<s>', 'luis'],
 ['se', '<unk>', 'por'],
 ['<unk>', 'váyanse', 'a'],
 ['q', 'todas', 'las'],
 ['<unk>', '<unk>', 'me'],
 ['vale', 'madres', '<unk>'],
 ['<unk>', 'crack', '<unk>'],
 ['me', 'pelan', 'la'],
 ['que', 'está', 'gorda'],
 ['hace', 'gorda', '<unk>'],
 ['<s>', 'ni', 'una'],
 ['<s>', '<s>', 'ama'],
 ['<s>', '<s>', '<s>'],
 ['estoy', '<unk>', 'la'],
 ['no', 'le', 'gusta'],
 ['<unk>', 'esos', 'camiones'],
 ['<unk>', 'militar', '<unk>'],
 ['bueno', 'si', 'estás'],
 ['la', 'madre', 'teresa'],
 ['putos', 'modos', '<unk>'],
 ['cantidad', 'de', 'personas'],
 ['son', 'las',

In [33]:
batch[1]

tensor([ 105,   38,    6,  475,  950, 4999, 3422,   33,  192, 1207,  160,   59,
          31,    1, 4997, 4997, 4999, 2377, 4999, 1024,   54, 4997,    8,    1,
        3285,  123, 4997, 4997,   44,  138, 3380, 4997, 2119,   28, 3449,   47,
         165, 1531, 1468,   38, 4997, 2508, 4997,  341,    6, 1953, 4999,   59,
          54,  389,  857,    6, 4997,   54,   32,   55, 4999, 4999,  718,    1,
          92,   44,  341,   59])

In [34]:
# Vocab size
args.vocab_size = ngram_data.get_vocab_size()

# Dimension of word embeddings
args.d = 50

# Dimension for hidden layer
args.d_h = 100

## Dropout
args.dropout = 0.1

In [39]:
class NeuralLM(nn.Module):
    
    def __init__(self, args, embeddings=None):
        super(NeuralLM, self).__init__()
        
        self.window_size = args.N-1
        self.embedding_dim = args.d
        
        self.emb = nn.Embedding(args.vocab_size, args.d)
        self.fc1 = nn.Linear(args.d*(args.N-1), args.d_h)
        self.drop1 = nn.Dropout(p=args.dropout)
        self.fc2 = nn.Linear(args.d_h, args.vocab_size, bias=False)
        
    def forward(self, x):
        x = self.emb(x)
        x = x.view(-1, self.window_size*self.embedding_dim)
        h = F.relu(self.fc1(x)) # relu(z) = max(0, z)
        h = self.drop1(h)
        return self.fc2(h)

In [40]:
def get_preds(raw_logits):
    probs = F.softmax(raw_logits.detach(), dim=1)
    y_pred = torch.argmax(probs, dim=1).cpu().numpy()
    
    return y_pred

In [41]:
def model_eval(data, model, gpu=False):
    with torch.no_grad():
        preds, tgts = [], []
        for window_words, labels in data:
            if gpu:
                window_words = window_words.cuda()

            outputs = model(window_words)

            # Get prediction
            y_pred = get_preds(outputs)

            tgt = labels.numpy()
            tgts.append(tgt)
            preds.append(y_pred)

        tgts = [e for l in tgts for e in l]
        preds = [e for l in preds for e in l]

        return accuracy_score(tgts, preds)

In [42]:
def save_checkpoint(state, is_best, checkpoint_path, filename="checkpoint.pt"):
    filename = os.path.join(checkpoint_path, filename)
    torch.save(state,filename)
    if is_best:
        shutil.copyfile(filename, os.path.join(checkpoint_path,"model_best.pt"))
     

In [ ]:
## Model Hyperparameters
args.vocab_size = ngram_data.get_vocab_size()
args.d = 100   # Dimension of word embeddings
args.d_h = 200 # Dimension for hidden layer
args.dropout = 0.1

# Training hyperparameters
args.lr = 2.3e-1
args.num_epochs = 100
args.patience = 20

## Scheduler hyperparameters
args.lr_patience = 10
args.lr_factor = 0.5

# Saving directory
args.savedir = 'model'
os.makedirs(args.savedir, exist_ok=True)

# Create model
model = NeuralLM(args)

# Send to GPU
args.use_gpu = torch.cuda.is_available()
if args.use_gpu:
    model.cuda()





# Loss, Optimizer and Scheduler
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.SGD(model.parameters(), lr=args.lr)

scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, 'min',
    patience=args.lr_patience,
    factor=args.lr_factor
)


In [46]:
start_time = time.time()
best_metric = 0
metric_history = []
train_metric_history = []
# n_no_improve = 0  # <-- inicializa el contador

for epoch in range(args.num_epochs):
    epoch_start_time = time.time()
    loss_epoch = []
    training_metric = []
    model.train()

    for window_words, labels in train_loader:

        # If GPU available
        if args.use_gpu:
            window_words = window_words.cuda()
            labels = labels.cuda()

        # Forward pass
        outputs = model(window_words)
        loss = criterion(outputs, labels)
        loss_epoch.append(loss.item())

        # Get training metrics
        y_pred = get_preds(outputs)
        tgt = labels.cpu().numpy()
        training_metric.append(accuracy_score(tgt, y_pred))

        # Backward and optimize
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

    # ----- TODO ESTO DEBE IR DENTRO DEL LOOP DE ÉPOCAS -----

    # Get metric in training dataset
    mean_epoch_metric = np.mean(training_metric)
    train_metric_history.append(mean_epoch_metric)

    # Get metric in validation dataset
    model.eval()
    tuning_metric = model_eval(val_loader, model, gpu=args.use_gpu)
    metric_history.append(tuning_metric)

    # Update scheduler
    scheduler.step(tuning_metric)

    # Check for metric improvement
    is_improvement = tuning_metric > best_metric
    if is_improvement:
        best_metric = tuning_metric
        n_no_improve = 0
    else:
        n_no_improve += 1

    # Save best model if metric improved
    save_checkpoint(
        {
            "epoch": epoch + 1,
            "state_dict": model.state_dict(),
            "optimizer": optimizer.state_dict(),
            "scheduler": scheduler.state_dict(),
            "best_metric": best_metric,
        },
        is_improvement,
        args.savedir,
    )

    # Early stopping
    if n_no_improve >= args.patience:
        print("No improvement. Breaking out of loop.")
        break
    
    print('Train acc: {:.4f}'.format(mean_epoch_metric))
    print('Epoch [{}/{}], Loss: {:.4f} - Val accuracy: {:.4f} - Epoch time: {:.2f}'
      .format(epoch + 1, args.num_epochs, np.mean(loss_epoch), tuning_metric, (time.time() - epoch_start_time)))
    
    
print('--- {:.2f} seconds ---'.format(time.time() - start_time))



Train acc: 0.1929
Epoch [1/100], Loss: 5.0256 - Val accuracy: 0.2081 - Epoch time: 11.56
Train acc: 0.1994
Epoch [2/100], Loss: 4.8172 - Val accuracy: 0.2409 - Epoch time: 11.10
Train acc: 0.2033
Epoch [3/100], Loss: 4.6488 - Val accuracy: 0.1595 - Epoch time: 11.12
Train acc: 0.2074
Epoch [4/100], Loss: 4.5051 - Val accuracy: 0.2214 - Epoch time: 11.26
Train acc: 0.2083
Epoch [5/100], Loss: 4.3755 - Val accuracy: 0.1786 - Epoch time: 11.45
Train acc: 0.2120
Epoch [6/100], Loss: 4.2529 - Val accuracy: 0.2139 - Epoch time: 11.31
Train acc: 0.2148
Epoch [7/100], Loss: 4.1313 - Val accuracy: 0.2138 - Epoch time: 11.32
Train acc: 0.2176
Epoch [8/100], Loss: 4.0336 - Val accuracy: 0.1957 - Epoch time: 11.38
Train acc: 0.2205
Epoch [9/100], Loss: 3.9256 - Val accuracy: 0.1440 - Epoch time: 11.25
Train acc: 0.2249
Epoch [10/100], Loss: 3.8340 - Val accuracy: 0.1381 - Epoch time: 11.25
Train acc: 0.2309
Epoch [11/100], Loss: 3.7455 - Val accuracy: 0.1742 - Epoch time: 11.31
Train acc: 0.2386
E

In [47]:
def print_closest_words(embeddings, ngram_data, word, n):
    word_id = torch.LongTensor([ngram_data.w2id[word]])      # get word id
    word_embed = embeddings(word_id)                         # get word embedding
    dists = torch.norm(embeddings.weight - word_embed, dim=1).detach()  # compute distances to all words
    lst = sorted(enumerate(dists.numpy()), key=lambda x: x[1])          # sort by distance

    for idx, difference in lst[1:n+1]:     # take the top n, ignore word itself
        print(ngram_data.id2w[idx], difference)


In [48]:
# Model with learned embeddings from scratch
best_model = NeuralLM(args)
best_model.load_state_dict(torch.load('model/model_best.pt')['state_dict'])
best_model.train(False)

print("-"*30)
print("Learned Embeddings")
print("-"*30)
print_closest_words(best_model.emb, ngram_data,"hijo",10)

------------------------------
Learned Embeddings
------------------------------
discurso 11.05926
durante 11.06263
carmona 11.215258
<unk> 11.311585
pongan 11.367791
explicar 11.398699
novia 11.4035425
licenciada 11.434308
pueblo 11.461182
🤦🏻‍♂ 11.552987


In [49]:
def parse_text(text, tokenizer):
    all_tokens = [w.lower() if w in ngram_data.w2id else '<unk>' for w in tokenizer.tokenize(text)]
    token_ids = [ngram_data.w2id[word.lower()] for word in all_tokens]
    return all_tokens, token_ids

In [50]:
def sample_next_word(logits, temperature=1.0):
    logits = np.asarray(logits).astype('float64')
    preds = logits / temperature
    exp_preds = np.exp(preds)
    preds = exp_preds / np.sum(exp_preds)
    probas = np.random.multinomial(1, preds)
    return np.argmax(probas)

In [51]:
def predict_next_token(model, token_ids):
    word_ids_tensor = torch.LongTensor(token_ids).unsqueeze(0)
    y_raw_pred = model(word_ids_tensor).squeeze(0).detach().numpy()

    # y_probs = F.softmax(y_raw_pred, dim=1)
    # y_pred = torch.argmax(y_probs, dim=1).detach().numpy()

    y_pred = sample_next_word(y_raw_pred, 1.0)
    return y_pred

In [52]:
def generate_sentence(model, initial_text, tokenizer):
    all_tokens, window_word_ids = parse_text(initial_text, tokenizer)

    for i in range(100):
        y_pred = predict_next_token(best_model, window_word_ids)
        next_word = ngram_data.id2w[y_pred]
        all_tokens.append(next_word)

        if next_word == '</s>':
            break
        else:
            window_word_ids.pop(0)
            window_word_ids.append(y_pred)

    return ' '.join(all_tokens)

In [53]:
initial_tokens = "<s> <s> <s>"

print("-"*30)
print("Learned Embeddings")
print("-"*30)
print(generate_sentence(best_model, initial_tokens, tk))

------------------------------
Learned Embeddings
------------------------------
<s> <s> <s> <unk> <unk> <unk> </s>


In [54]:
initial_tokens = "<s> <s> estoy"

print("-"*30)
print("Learned embeddings")
print("-"*30)
print(generate_sentence(best_model, initial_tokens, tk))

------------------------------
Learned embeddings
------------------------------
<s> <s> estoy pasaron por qué <unk> están <unk> <unk> 😂 <unk> sigues banca secretos a doy que <unk> <unk> loca <unk> es que putas y chuky </s>


In [55]:
initial_tokens = "<s> saludos a"

print("-"*30)
print("Learned embeddings")
print("-"*30)
print(generate_sentence(best_model, initial_tokens, tk))

------------------------------
Learned embeddings
------------------------------
<s> saludos a caras <unk> <unk> merece fue <unk> que llegas <unk> chinguen a pesar <unk> </s>


In [56]:
initial_tokens = "yo opino que"

print("-"*30)
print("Learned embeddings")
print("-"*30)
print(generate_sentence(best_model, initial_tokens, tk))

------------------------------
Learned embeddings
------------------------------
yo opino que castra <unk> <unk> no mamones que salió su <unk> cansaste <unk> <unk> <unk> no cara dieron <unk> <unk> « <unk> <unk> en el <unk> de <unk> análisis y cuando son verga <unk> ☀ un <unk> de <unk> 😙 que <unk> su mierda y amor de <unk> está muy pero <unk> loca <unk> a veces te <unk> <unk> <unk> <unk> de <unk> al mundo $ <unk> </s>


In [57]:
def log_likelihood(model, text, ngram_model):
    # Generate n-gram windows from input text and the respective label y
    X, y = ngram_data.transform([text])
    # Discard first two n-gram windows since they contain '<s>' tokens not necessary
    X, y = X[2:], y[2:]
    X = torch.LongTensor(X).unsqueeze(0)

    logits = model(X).detach()
    probs = F.softmax(logits, dim=1).numpy()

    return np.sum([np.log(probs[i][w]) for i, w in enumerate(y)])

In [58]:
print("Log Likelihood: ",log_likelihood(best_model,"Estamos en la clase de procesamiento de lenguaje",ngram_data))
print("Log Likelihood: ",log_likelihood(best_model,"Estamos procesamiento clase en la de natural de lenguaje",ngram_data))
print("Log Likelihood: ",log_likelihood(best_model,"la natural Estamos clase en de de lenguaje procesamiento",ngram_data))

Log Likelihood:  -18.537767
Log Likelihood:  -42.07772
Log Likelihood:  -40.183098


In [59]:
from itertools import permutations
from random import shuffle

word_list = "sino gano me voy a la chingada".split(' ')
perms = [' '.join(perm) for perm in permutations(word_list)]
# print(len(perms))


print('-'*50)
for p, t in sorted([(log_likelihood(best_model, text, ngram_data), text) for text in perms],reverse=True)[:5]:
    print(p, t)
    
print('-'*50)

for p, t in sorted([(log_likelihood(best_model, text, ngram_data), text) for text in perms],reverse=True)[-5:]:
    print(p, t)


--------------------------------------------------
-19.90253 gano sino me voy a la chingada
-20.632774 sino gano me voy a la chingada
-23.896263 gano me voy a la chingada sino
-24.898602 sino me voy a la chingada gano
-25.30316 gano voy me sino a la chingada
--------------------------------------------------
-55.70315 me a gano voy chingada sino la
-55.717 a la voy gano chingada sino me
-56.30478 a la gano voy chingada sino me
-57.141483 la a voy gano chingada sino me
-57.68225 la a gano voy chingada sino me
